In [ ]:
!pip install plotly pandas requests nbformat -q

import requests
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta

API_URL = 'https://djtnqbvkhqftmtnsityx.supabase.co/rest/v1/'
API_KEY = 'sb_publishable_NnOjc1iJWmA7j418h79mEg_AHh9h49u'
HEADERS = {'apikey': API_KEY, 'Authorization': f'Bearer {API_KEY}'}

def query_supabase(endpoint, select='*', filters=None, limit=10000):
    """Query Supabase REST API with filters."""
    params = {}
    if select: params['select'] = select
    if filters:
        for k, v in filters.items():
            params[k] = f'eq.{v}'
    if limit: params['limit'] = limit
    r = requests.get(f'{API_URL}{endpoint}', headers=HEADERS, params=params)
    if r.status_code == 200:
        return pd.DataFrame(r.json())
    else:
        print(f'Error {r.status_code}: {r.text[:200]}')
        return pd.DataFrame()

In [ ]:
# Query daily funding for all venues/symbols with yield data
df = query_supabase(
    'daily_funding',
    select='venue,symbol,date,avg_rate_bps,daily_annualized_yield_pct,asset_class',
    limit=50000
)

df['date'] = pd.to_datetime(df['date'])
df['avg_rate_bps'] = pd.to_numeric(df['avg_rate_bps'], errors='coerce')
df['daily_annualized_yield_pct'] = pd.to_numeric(df['daily_annualized_yield_pct'], errors='coerce')
df = df.dropna(subset=['daily_annualized_yield_pct'])

print(f"Loaded {len(df):,} rows with annualized yield data")
print(f"Venues: {df['venue'].unique().tolist()}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
df.head()

In [ ]:
# Line chart: annualized yield over time by venue
yield_daily = df.groupby(['date', 'venue'])['daily_annualized_yield_pct'].mean().reset_index()

fig = px.line(
    yield_daily, x='date', y='daily_annualized_yield_pct', color='venue',
    title='Annualized Funding Yield by Venue Over Time',
    labels={'daily_annualized_yield_pct': 'Annualized Yield (%)', 'date': 'Date', 'venue': 'Venue'},
    template='plotly_dark'
)
fig.update_layout(height=500, hovermode='x unified')
fig.show()

In [ ]:
# Scatter: yield vs rate volatility (std dev) by venue-symbol
scatter_df = df.groupby(['venue', 'symbol']).agg(
    avg_yield=('daily_annualized_yield_pct', 'mean'),
    yield_std=('daily_annualized_yield_pct', 'std'),
    avg_rate=('avg_rate_bps', 'mean'),
    observations=('date', 'count')
).reset_index().dropna()

fig = px.scatter(
    scatter_df, x='yield_std', y='avg_yield', color='venue', size='observations',
    hover_data=['symbol', 'avg_rate'],
    title='Yield vs Volatility by Venue-Symbol',
    labels={'yield_std': 'Yield Std Dev (%)', 'avg_yield': 'Avg Annualized Yield (%)', 'venue': 'Venue'},
    template='plotly_dark'
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Area chart: cumulative yield if held continuously for 1 year by venue
yield_daily = df.groupby(['date', 'venue'])['daily_annualized_yield_pct'].mean().reset_index()
yield_daily = yield_daily.sort_values('date')

# Cumulative yield = sum of daily yield / 365 * 100 (convert pct to cumulative)
yield_daily['daily_yield_frac'] = yield_daily['daily_annualized_yield_pct'] / 365 / 100
yield_daily['cumulative_yield_pct'] = yield_daily.groupby('venue')['daily_yield_frac'].cumsum() * 100

fig = px.area(
    yield_daily, x='date', y='cumulative_yield_pct', color='venue',
    title='Cumulative Yield (If Held Continuously)',
    labels={'cumulative_yield_pct': 'Cumulative Yield (%)', 'date': 'Date', 'venue': 'Venue'},
    template='plotly_dark'
)
fig.update_layout(height=500, hovermode='x unified')
fig.show()